# 16j — antibody prevalence around one forecast origin: framework input vs ONS observation

14j §6c draws antibody prevalence over the whole CoMix span. This notebook zooms to **one forecast
origin — 10j's, 2021-05-09 — and to exactly the weeks the framework touches there**: the `smax = 4`
renewal-lag weeks, the `n_fit = 8` fit weeks and the 4 forecast-target weeks of
`WeeklyWindow(origin; n_fit, smax, horizons)`. In those weeks it overlays, per CIS age bin:

| series | what it is |
|---|---|
| **squares** | the weekly antibody covariate `A_a(t)` the **framework builds from `estimates_age_ab.csv`** (`load_window_data` → `weekly_antibody`): `wd.antibody` over lag + fit weeks, `wd.antibody_fc` at the forecast targets — what entered the NGM's leaky factor `(1 + (F−1)·A_a(t))` while `_EST_PATH` named the `_ab` file |
| orange line + band | the **daily `gen_dab`** those squares average (median, 90% CrI; mean dashed) |
| red points + bars | the **ONS antibody positivity the `_ab` fit was trained on** (179 ng/ml, vintage derived by `ons_antibody_fit_vintage`), rebinned to the CIS bins exactly as inc2prev's `read_ab()` does |
| **right axis:** black line + circles | **observed weekly infections** over all 16 weeks — `estimates_age.csv` (`_EST_PATH`), the truth 9j scored the forecasts against |
| **right axis:** coloured dashed ◇ + faint bands | each of 10j's four models' **calibration** — its in-sample fitted incidence over the 8 fit weeks (median, 90% CrI), rebuilt read-only from the h=1 Stage-1 chain + Stage-2 pooled artefacts by 10j's `fit_window_infection_draws` |
| **right axis:** coloured solid ○ + bands | the same models' **forecast from this origin** (median, 90% predictive band) over the 4 targets — the live grid's pooled fans, read from 9j's assembly cache |

⚠ **The two model bands are different quantities** (as in 10j §1). Calibration is the MEAN of Stage
2's infection likelihood given the OBSERVED lags — a one-step-ahead fitted mean with no observation
noise. The forecast is posterior-PREDICTIVE and self-iterated from the origin. A narrow dashed band
next to a wide solid one is expected.

⚠ **The incidence and the antibody come from DIFFERENT inc2prev files.** The forecasts are the live
grid's, fitted to `estimates_age.csv`, so their comparator is that file's infections. The antibody is
`estimates_age_ab.csv`'s, the only file that carries any. With `F = 1` antibody never entered these
forecasts: the overlay shows *timing*, not a causal link.

⚠ **inc2prev models the proportion antibody-POSITIVE, not a titre** — the quantity ONS publishes.

⚠ **Inert today.** `_EST_PATH` names `estimates_age.csv` (no `gen_dab` ⇒ `A ≡ 0`) and
`model_transmission` pins `F = 1.0`, so `(1 + (F−1)·A) ≡ 1`. This notebook reads the `_ab` file
explicitly: it shows what the previous Stage-2 generation (`dt_intermediate_residual/`) was handed,
and what re-enabling `F` would feed — not anything the live grid uses.

⚠ **The forecast-week `A` is the REALISED future value** (`gen_dab` at t₀+h), the same
contact-updated-iterate convention as `Cstar_end`. It is not a forecast of antibody.

**Cannot refit.** The forecast fans come from 9j's assembly cache through 15j's `load_9j_assembly`
(`isfile` → `load` → validate), which never calls `assemble_or_load_forecasts`. The calibration
comes from `fit_window_infection_draws`, which is `isfile` → `load` → replay and returns `nothing`
on a miss — the cell asserts all four are present, and that every artefact exists before reading.
Neither path calls `two_stage_forecast` or `fit_or_load_*`. Outputs go to `res/16j_*`.


In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # framework: WeeklyWindow, FrameworkConfig, load_window_data, …
include("9j_viz_utils.jl")     # panel_grid (and 8j, loaded once through it)
include("10j_viz_utils.jl")    # fit_window_infection_draws — the in-sample calibration fans
include("14j_viz_utils.jl")    # ons_antibody_age / _fit_vintage / _to_cis, i2p_gen_dab, _i2p_est_path
include("15j_viz_utils.jl")    # load_9j_assembly / assembly_truth — the read-only forecast cache
include("16j_viz_utils.jl")    # this notebook's helpers
using Statistics, DataFrames, Printf, CSV
mkpath("../res");


In [ ]:
# 10j's window, reproduced from 10j's own construction: same cfg defaults, same origin, same
# `WeeklyWindow(o; n_fit, smax, horizons)`. 10j additionally asserts ORIGIN ∈ available forecast
# origins, which needs the CoMix read; the window itself is a pure function of (ORIGIN, cfg), so
# the Sunday alignment and the segment lengths are asserted instead.
cfg    = FrameworkConfig(constant_contacts = false)
grid   = cis_age_grid()
ORIGIN = Date(2021, 5, 10)                      # 10j cell 2
win    = WeeklyWindow(ORIGIN; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)

@assert week_start(ORIGIN) == ORIGIN "ORIGIN must be a Sunday week start"
@assert (length(win.lag_weeks), length(win.fit_weeks), length(win.forecast_weeks)) ==
        (cfg.smax, cfg.n_fit, length(cfg.horizons)) == (4, 8, 4) "not 10j's 4 + 8 + 4 window"
@printf("origin          %s\nlag weeks       %s … %s\nfit weeks       %s … %s\nforecast weeks  %s … %s\n",
        ORIGIN, first(win.lag_weeks), last(win.lag_weeks), first(win.fit_weeks), last(win.fit_weeks),
        first(win.forecast_weeks), last(win.forecast_weeks))


In [ ]:
# The `_ab` estimates, read EXPLICITLY (never through `_EST_PATH`, which names the antibody-free
# file), and turned into WindowData by the framework's own `load_window_data`.
EST_AB      = _i2p_est_path("estimates_age_ab")
df_ab, tmap = _load_estimates(EST_AB)
wd          = load_window_data(win, df_ab, tmap; grid = grid)
GD          = i2p_gen_dab(; grid = grid)        # daily gen_dab; asserts one shared t_index map

# TRIPWIRE: the framework's A must be EXACTLY the weekly mean of gen_dab's `mean` column, recomputed
# here independently of `weekly_antibody`. Any drift means week bucketing or bin mapping moved.
_recomp(weeks) = [mean(GD.mean[(GD.bin .== a) .& (week_start.(GD.date) .== w)])
                  for a in 1:grid.N, w in weeks]
@assert _recomp(win.all_weeks) == wd.antibody      "wd.antibody ≠ weekly mean of gen_dab"
@assert _recomp(win.forecast_weeks) == wd.antibody_fc "wd.antibody_fc ≠ weekly mean of gen_dab"
println("tripwire: wd.antibody / antibody_fc reproduce the weekly mean of gen_dab EXACTLY")

# The ONS series the `_ab` fit read, rebinned to the CIS bins as read_ab() does.
V_FIT = ons_antibody_fit_vintage()
OBS   = ons_antibody_to_cis(ons_antibody_age(vintage = V_FIT, thresholds = ("standard",)); grid = grid)
@printf("ONS vintage the fit read: %s (179 ng/ml); %d CIS-bin weeks\n", V_FIT, nrow(OBS))

# ── INCIDENCE (right axis) ──────────────────────────────────────────────────────────────────────
# The forecasts are the LIVE grid's, read from 9j's assembly cache — never refitted. `LABELS` is 9j's
# six-model order (the cache validates it); FC_LABELS are the ones drawn: 10j's four, matching the
# window. Append a label to draw it too.
@assert cfg.stage1_use_nuts && contacts_label(cfg) == "temporal-w8h-lc0-eng-nuts"
combos = vcat([(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                        for nb in (MeanNGM(), NeighbourhoodDegreeNGM())],
              [(NegBinAgePair(), DiagonalMeanNGM()), (NoContactDegree(), NullNGM())])
LABELS    = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]
FC_LABELS = LABELS[1:4]
asm = load_9j_assembly(cfg; labels = LABELS, grid = grid)
tru = assembly_truth(asm.qall, grid; horizons = cfg.horizons)
@assert ORIGIN in asm.origins "ORIGIN $(ORIGIN) is not in the 9j grid"

# Observed infections from the file the forecasts were fitted to (`_EST_PATH`), NOT the `_ab` one.
wd_live = load_window_data(win; grid = grid)
truth   = load_forecast_truth(win; grid = grid)
# ⚠ `load_9j_assembly` may WARN that the cache is stale: it compares MODIFICATION TIMES, and a
# checkout of the inc2prev submodule touches `estimates_age.csv` without changing it (verified
# 2026-10-16: no diff against the committed file). The assert below compares CONTENTS at this
# origin, which is the check that matters here — it supersedes the mtime warning.
# TRIPWIRE: the truth 9j scored against must be exactly this file's — else the black line and the
# fans are on different definitions of "infections".
@assert tru.truth[ORIGIN] == truth "the 9j cache's truth ≠ load_forecast_truth — stale cache?"
println("tripwire: the 9j cache's truth at $(ORIGIN) equals load_forecast_truth EXACTLY")

# ── CALIBRATION (right axis, fit weeks) ─────────────────────────────────────────────────────────
# 10j §1's in-sample fit: per model, the h=1 Stage-1 chain is replayed for its per-week C*, paired
# with the Stage-2 pooled draws, and pushed through ONE renewal step with the OBSERVED lags. It needs
# the h=1 contact window's degree data (one CoMix read) and `wd_live` — the estimates the Stage-2
# fits were made against (`_EST_PATH`), NOT the `_ab` file.
# GUARD FIRST, as 10j cell 4 does: every artefact must exist. `fit_window_infection_draws` cannot
# fit, but a miss returns `nothing` and the panel would silently lose that model.
_tag  = contacts_label(cfg)
_miss = [basename(p) for (dm, nb) in combos[1:4]
         for p in (stage1_chain_path(string(degree_label(dm), "|", ngm_label(nb)), ORIGIN, 1;
                                     contacts = _tag, save_dir = "../dt_intermediate"),
                   stage2_pooled_path(string(degree_label(dm), "|", ngm_label(nb)), ORIGIN, 1;
                                      contacts = _tag, save_dir = "../dt_intermediate"))
         if !isfile(p)]
@assert isempty(_miss) "h=1 artefacts missing for $(ORIGIN): $(_miss)"

raw  = load_raw_contact_inputs()
apd1 = prepare_degree_data(degree_window(ORIGIN, 1, cfg), cfg; grid = grid, setting = :all,
                           df_part_raw = raw.df_part, craw_raw = raw.craw)
CAL_STORE = Dict{String,Array{Float64,3}}()
for (dm, nb) in combos[1:4]
    r = fit_window_infection_draws(dm, nb, [apd1], wd_live, cfg, ORIGIN; h = 1,
                                   save_dir = "../dt_intermediate")
    @assert r !== nothing "calibration draws failed for $(degree_label(dm))|$(ngm_label(nb)) — grep the log for `could not load`"
    CAL_STORE[string(degree_label(dm), "|", ngm_label(nb))] = r
end
@assert Set(keys(CAL_STORE)) == Set(FC_LABELS)
println("calibration: in-sample draws for ", join(FC_LABELS, ", "), " — ",
        join(size(first(values(CAL_STORE))), " × "))


## §1 — The window, per age bin

Background bands mark the three `WeeklyWindow` segments; the dashed line is where the forecast
starts (the end of origin week t₀). Each panel has its own y-range (the bins sit anywhere from ~20%
to ~95% in these 16 weeks, so a shared 0–100 axis would flatten every weekly movement).

⚠ **Two calendars, one day apart.** ONS weeks run Monday–Sunday and are drawn at their Thursday
mid-date; framework weeks run Sunday–Saturday and are drawn at their Wednesday `week_mid`. An ONS
week is matched to the framework week containing its Thursday (6 of 7 days shared), asserted
one-to-one in `antibody_window_frame`.

⚠ **Squares sit on the MEAN, not the median.** `weekly_antibody` averages `gen_dab`'s posterior
**mean**; the child posteriors are right-skewed, so there the squares sit ~3 pp above the median
line — on the dashed mean line.


In [ ]:
FR = antibody_window_frame(win, wd, OBS; grid = grid)
SM = antibody_window_summary(FR)
IO = incidence_window_frame(win, wd_live, truth; grid = grid)
IF = forecast_window_frame(asm.fc_store, ORIGIN, FC_LABELS, win; grid = grid)
IC = calibration_window_frame(CAL_STORE, FC_LABELS, win; grid = grid)
CSV.write("../res/16j_antibody_window.csv", FR)
CSV.write("../res/16j_antibody_window_summary.csv", SM)
CSV.write("../res/16j_incidence_window.csv", IO)
CSV.write("../res/16j_forecast_window.csv", IF)
CSV.write("../res/16j_calibration_window.csv", IC)

display(make_antibody_window_fig(FR, GD, win; grid = grid, inc_obs = IO, inc_fc = IF, inc_cal = IC,
                                 fc_labels = FC_LABELS, res_dir = "../res"))

# Framework A minus ONS, per bin × segment. Printed rather than displayed so no column is dropped.
@printf("%-6s %-9s %4s %12s %13s %12s\n", "bin", "segment", "n", "median Δ pp", "median |Δ| pp",
        "outside 95%")
for r in eachrow(SM)
    r.n_matched == 0 ? @printf("%-6s %-9s %4d %12s %13s %12s\n", r.bin_label, r.segment, 0, "—", "—", "no ONS") :
        @printf("%-6s %-9s %4d %+12.2f %13.2f %9d/%-2d\n", r.bin_label, r.segment, r.n_matched,
                r.median_diff_pp, r.median_absdiff_pp, r.n_outside_95, r.n_matched)
end

# Forecast vs observed infections at the 4 targets: median ratio forecast/observed, and how many of
# the 4 weeks the observation falls inside the model's 90% band. Per bin × model.
println()
@printf("%-6s %-32s %s\n", "bin", "model", "forecast median ÷ observed, h = 1..4      inside 90%")
for a in 1:grid.N, l in FC_LABELS
    q = sort(IF[(IF.bin .== a) .& (IF.label .== l), :], :h)
    o = truth[a, :]
    @printf("%-6s %-32s %s   %d/4\n", grid.LAB[a], l, join([@sprintf("%5.2f", r) for r in q.med ./ o], " "),
            count(q.lo .<= o .<= q.hi))
end

# Calibration vs observed over the 8 fit weeks: median ratio fitted/observed and how many weeks the
# observation sits inside the in-sample 90% band (narrow by construction — see the caption).
println()
@printf("%-6s %-32s %s\n", "bin", "model", "calibration: median fitted ÷ observed (8 fit wk)   inside 90%")
obs_fit = wd_live.I_mean[:, (cfg.smax + 1):end]
for a in 1:grid.N, l in FC_LABELS
    c = sort(IC[(IC.bin .== a) .& (IC.label .== l), :], :week)
    o = obs_fit[a, :]
    @printf("%-6s %-32s %6.2f  (range %4.2f – %4.2f)                    %d/8\n", grid.LAB[a], l,
            median(c.med ./ o), minimum(c.med ./ o), maximum(c.med ./ o), count(c.lo .<= o .<= c.hi))
end


**What to look for.**

- **In the fit window the framework's `A` tracks ONS to ~2–3 pp in `16-24` and `35-49`** (no week
  outside the published CI), sits **~5 pp high in `25-34`**, and runs **BELOW the observation in the
  two oldest bins** — `50-69` by a median 8.3 pp (7 of 8 weeks outside the CI) and `70+` by 4.4 pp.
  These are the weeks the vaccination roll-out was driving those bins up: the modelled rise is too
  shallow, the pattern 14j §6c found over the whole span.
- **In the forecast weeks `35-49` drops 11.6 pp below ONS, all four weeks outside the CI**: the
  observed climb from ~65% to ~87% over those four weeks runs ahead of `gen_dab`.
  Because this column is the REALISED t₀+h value, that is the antibody the NGM would have been
  handed for the forecast, too low exactly where protection was arriving fastest.
- **Direction of the error if `F` were switched back on**: too-low `A` means too little antibody
  protection, so `(1 + (F−1)·A)` with `F < 1` would leave those bins **too susceptible** and the NGM
  would over-weight transmission into `35-69` around this origin.
- **Right axis, calibration — the fit itself is good in the middle adult bins.** Over the 8 fit
  weeks the in-sample median sits at **0.89–1.01× the observation in `16-24`, `25-34` and `35-49`**,
  lower in the children and `50+` (0.69–0.97×), with the widest misses in the first fit weeks (one
  model's `11-15` week-1 spike reaches 1.75×). The observation falls inside the narrow in-sample band
  in 0–7 of 8 weeks; that band carries no observation noise, so a miss there is not a calibration
  failure by itself.
- **Right axis, forecast — every model under-forecasts this origin, and the gap widens with the
  horizon.** Set against the ~0.9–1.0× calibration above, that places the failure in the FORECAST,
  not the fit.
  At h=4 the forecast medians are 6–56% of the observed infections; the observation falls inside a
  model's 90% band in at most 2 of the 4 target weeks, and in none for `11-15`, `70+` and most
  `50-69` cells. The forecasts decline while observed infections turn upward, steepest in `16-24`
  (12 k → 49 k a week over the four targets) and `25-34` (8 k → 24 k): the turn of the epidemic in
  late May 2021, which a renewal forecast frozen at the origin week cannot anticipate.
- **Within each degree model the neighbourhood (excess-degree) NGM sits closer to the observation
  than the mean NGM** in most cells, the ordering 9j's WIS reports over the whole grid.
- **Timing against the antibody**: the sharpest incidence rise is in the bins with the LOWEST
  antibody prevalence (`16-24` at ~42–46%, `25-34` ~49–55% over the targets) while the >=90% bins
  (`70+`, late `50-69`) stay comparatively flat. That is consistent with partial protection, but with
  `F = 1` none of these forecasts could use it — the overlay shows coincidence in time, not a fitted
  effect.
- **The child bins (`2-10`, `11-15`) are model only**, with the widest bands and the mean-vs-median
  offset described above. They borrow `16-24`'s seroconversion and waning parameters and have no
  ONS data, so nothing here can check them.


## Outputs

Written to `../res`:

| file | what |
|---|---|
| `16j_antibody_window.png` | §1's figure — one panel per CIS bin, the 4 + 8 + 4 weeks of 10j's window |
| `16j_antibody_window.csv` | the (bin × week) frame behind it: segment, framework `A`, matched ONS value and CI |
| `16j_incidence_window.csv` | observed weekly infections per bin over the 16 weeks (`estimates_age.csv`), with the segment |
| `16j_calibration_window.csv` | each drawn model's in-sample fitted incidence over the 8 fit weeks: 5% / median / 95% |
| `16j_forecast_window.csv` | each drawn model's forecast at the 4 targets: 5% / median / 95% over its 10 000 pooled draws |
| `16j_antibody_window_summary.csv` | per bin × segment: matched weeks, median signed and absolute Δ (pp), weeks outside the ONS 95% CI |

To look at another origin, change `ORIGIN` in the config cell; the window, the frames and the figure
all derive from it. It must be one of the 9j grid's 67 origins (2020-10-19 … 2022-01-24) — the
forecasts come from that cache, and the cell asserts membership. To draw other models, edit
`FC_LABELS` (any of `LABELS`).


(§2 — the England SGTF / S-positive share — and §3 — the ONS variant mix across the CoMix span — were
removed on 2026-10-22 with their data folder `dt_ons_variant/`: variant proportions now come from
COG-UK alone, `dt_cog_uk/`; 1j draws them.)